<a href="https://colab.research.google.com/github/TirthankaSaha/AI-powered-Virtual-Development-Pod/blob/main/SDLC_Pod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install openai==0.28
!pip install gradio PyMuPDF openai fpdf

# Imports
import gradio as gr
import fitz  # PyMuPDF
import openai
import subprocess
import tempfile
import os
from fpdf import FPDF

# OpenRouter API setup
openai.api_key = "Your OpenRouter API key"
openai.api_base = "https://openrouter.ai/api/v1"

# Session state
session_state = {
    "user_stories": None,
    "design_doc": None,
    "generated_code": None,
    "test_cases": None
}

# PDF Text Extraction
def extract_text_from_pdf(pdf_file):
    doc = fitz.open(pdf_file.name)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

# PDF Export Functionality
def export_to_pdf():
    pdf = FPDF()
    pdf.set_auto_page_break(auto=True, margin=15)
    pdf.add_page()
    pdf.set_font("Arial", size=12)

    sections = {
        "User Stories": session_state.get("user_stories", "No user stories generated."),
        "System Design": session_state.get("design_doc", "No design document generated."),
        "Backend Code": session_state.get("generated_code", "No code generated."),
        "Test Cases": session_state.get("test_cases", "No test cases generated.")
    }

    for title, content in sections.items():
        pdf.set_font("Arial", 'B', 14)
        pdf.cell(200, 10, txt=title, ln=True)
        pdf.set_font("Arial", size=10)
        for line in content.split("\n"):
            pdf.multi_cell(0, 5, txt=line)
        pdf.ln(5)

    temp_file_path = os.path.join(tempfile.gettempdir(), "virtual_dev_pod_output.pdf")
    pdf.output(temp_file_path)
    return temp_file_path

# Business Analyst Agent
def business_analyst_agent(rfp_pdf):
    try:
        rfp_text = extract_text_from_pdf(rfp_pdf)
        prompt = f"""
You are a Business Analyst AI agent. Read the following RFP content and generate 10 clear, numbered user stories based on the requirements provided. Each user story should follow the format: "As a [type of user], I want to [goal] so that [reason]."

RFP:
{rfp_text}

Output:
"""
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[
                {"role": "system", "content": "You are an expert business analyst."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        user_stories = response['choices'][0]['message']['content']
        session_state["user_stories"] = user_stories
        return user_stories
    except Exception as e:
        return f"❌ Error: {str(e)}"

# Design Agent
def design_agent(_):
    try:
        user_stories = session_state["user_stories"]
        if not user_stories:
            return "❌ No user stories available."
        prompt = f"""
You are a software design expert AI. Based on the following user stories, generate a detailed system design document. Include:

- Key components and architecture
- Database schema (in simple table format)
- API endpoints
- User roles and access control

User Stories:
{user_stories}

Design Document:
"""
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[
                {"role": "system", "content": "You are a software design expert."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7
        )
        design_doc = response['choices'][0]['message']['content']
        session_state["design_doc"] = design_doc
        return design_doc
    except Exception as e:
        return f"❌ Error in Design Agent: {str(e)}"

# Code Agent
def code_agent(_):
    try:
        user_stories = session_state["user_stories"]
        design_doc = session_state["design_doc"]
        if not user_stories or not design_doc:
            return "❌ Missing user stories or design doc."
        prompt = f"""
You are a senior backend developer AI. Based on the following user stories and design document, generate production-ready Python backend code using FastAPI.

Make sure the code includes:
- Endpoint definitions
- Business logic
- Comments for clarity
- Basic data models (e.g., Pydantic)

User Stories:
{user_stories}

Design Document:
{design_doc}

Generated Code:
"""
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[
                {"role": "system", "content": "You are a senior backend developer."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.6
        )
        generated_code = response['choices'][0]['message']['content']
        session_state["generated_code"] = generated_code
        return generated_code
    except Exception as e:
        return f"❌ Error in Code Agent: {str(e)}"

# Testing Agent
def testing_agent(_):
    try:
        user_stories = session_state["user_stories"]
        generated_code = session_state["generated_code"]
        if not user_stories or not generated_code:
            return "❌ Missing user stories or code."
        prompt = f"""
You are a QA testing expert AI. Based on the following user stories and backend code, generate Python unit test cases using the `pytest` framework.

Ensure the tests:
- Cover key functionality described in user stories
- Use realistic input/output examples
- Are well-structured and easy to run

User Stories:
{user_stories}

Generated Code:
{generated_code}

Now generate the test cases:
"""
        response = openai.ChatCompletion.create(
            model="mistralai/mixtral-8x7b-instruct",
            messages=[
                {"role": "system", "content": "You are an expert software QA tester."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.5
        )
        test_cases = response['choices'][0]['message']['content']
        session_state["test_cases"] = test_cases
        return test_cases
    except Exception as e:
        return f"❌ Error in Testing Agent: {str(e)}"

# Run tests
def run_tests(_):
    try:
        code = session_state["generated_code"]
        tests = session_state["test_cases"]
        if not code or not tests:
            return "❌ Missing code or test cases."

        with tempfile.TemporaryDirectory() as tmpdirname:
            code_file = os.path.join(tmpdirname, "app.py")
            test_file = os.path.join(tmpdirname, "test_app.py")
            with open(code_file, "w") as f:
                f.write(code)
            with open(test_file, "w") as f:
                f.write(tests)

            result = subprocess.run(
                ["pytest", test_file, "--tb=short", "-q"],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                cwd=tmpdirname
            )
            return f"✅ Test Results:\n\n{result.stdout}"
    except Exception as e:
        return f"❌ Error while running tests: {str(e)}"

# Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🤖 NEXUS : AI-Powered Virtual Development Pod

    🚀 This virtual pod simulates a complete software development workflow using AI agents:
    - 🧠 Business Analyst (User Stories)
    - 🏗️ Design Engineer (System Design)
    - 👨‍💻 Developer (Backend Code)
    - 🧪 QA Tester (Test Cases)
    - ✅ Test Runner (Execution Output)
    - 📄 Export Output to PDF

    Upload your RFP PDF and let the agents handle the rest!
    """)

    with gr.Tabs():
        with gr.Tab("1️⃣ Business Analyst Agent"):
            with gr.Row():
                with gr.Column(scale=1):
                    rfp_input = gr.File(label="📄 Upload RFP PDF", file_types=[".pdf"])
                    submit_btn = gr.Button("✨ Generate User Stories")
                with gr.Column(scale=2):
                    output = gr.Textbox(label="📋 Generated User Stories", lines=20, max_lines=30, interactive=False)
            submit_btn.click(fn=business_analyst_agent, inputs=[rfp_input], outputs=[output])

        with gr.Tab("2️⃣ Design Agent"):
            with gr.Row():
                with gr.Column(scale=1):
                    design_btn = gr.Button("🛠️ Generate Design Document")
                with gr.Column(scale=2):
                    design_output = gr.Textbox(label="📐 System Design", lines=20, max_lines=30, interactive=False)
            design_btn.click(fn=design_agent, inputs=[], outputs=[design_output])

        with gr.Tab("3️⃣ Coding Agent"):
            with gr.Row():
                with gr.Column(scale=1):
                    code_btn = gr.Button("💻 Generate Code")
                with gr.Column(scale=2):
                    code_output = gr.Code(label="🧩 Backend Code (FastAPI)", language="python", lines=20)
            code_btn.click(fn=code_agent, inputs=[], outputs=[code_output])

        with gr.Tab("4️⃣ Testing Agent"):
            with gr.Row():
                with gr.Column(scale=1):
                    test_btn = gr.Button("🔍 Generate Test Cases")
                with gr.Column(scale=2):
                    test_output = gr.Code(label="🧪 Pytest Test Cases", language="python", lines=20)
            test_btn.click(fn=testing_agent, inputs=[], outputs=[test_output])

        with gr.Tab("5️⃣ Test Runner"):
            with gr.Row():
                with gr.Column(scale=1):
                    run_test_btn = gr.Button("🏁 Run Tests")
                with gr.Column(scale=2):
                    test_result_output = gr.Textbox(label="✅ Test Run Output", lines=20, max_lines=30, interactive=False)
            run_test_btn.click(fn=run_tests, inputs=[], outputs=[test_result_output])

        with gr.Tab("6️⃣ Summary PDF"):
            with gr.Row():
                with gr.Column(scale=1):
                    export_btn = gr.Button("📤 Export All to PDF")
                with gr.Column(scale=2):
                    pdf_file_output = gr.File(label="📄 Download PDF")
            export_btn.click(fn=export_to_pdf, inputs=[], outputs=[pdf_file_output])

demo.launch()

/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1024: UserWarning: Expected 1 arguments for function <function design_agent at 0x7f83585a4ae0>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1028: UserWarning: Expected at least 1 arguments for function <function design_agent at 0x7f83585a4ae0>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1024: UserWarning: Expected 1 arguments for function <function code_agent at 0x7f83585a6200>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1028: UserWarning: Expected at least 1 arguments for function <function code_agent at 0x7f83585a6200>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1024: UserWarning: Expected 1 arguments for function <function testing_agent at 0x7f83585a62a0>, received 0.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/gradio/utils.py:1028: UserWarning: Expec

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://17f2d8e22816ac3cbd.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
